# 08 - Cross-Dataset Result Synthesis

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Notebook tổng hợp CSV/JSON từ các notebook trước, không train model và không thay đổi threshold.
Attach outputs của Notebook 02-07 bằng Add Input khi chạy trên Kaggle.

In [ ]:
from src.artifacts import find_result_file, load_frozen_reference_artifact, sha256_file
from src.data import load_config

ieee_config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
baf_config = load_config(PROJECT_ROOT / "configs/baf.yaml")
ieee_artifact = load_frozen_reference_artifact("ieee_cis", expected_config=ieee_config, search_roots=INPUT_ROOTS)
baf_artifact = load_frozen_reference_artifact("baf", expected_config=baf_config, search_roots=INPUT_ROOTS)
ieee_benchmark_root = ieee_artifact["manifest_path"].parent
baf_benchmark_root = baf_artifact["manifest_path"].parent
output_dir = OUTPUT_BASE / "08_cross_dataset_result_synthesis"
output_dir.mkdir(parents=True, exist_ok=True)

required = {
    "ieee_predictive": ieee_benchmark_root / "predictive_metrics_summary.csv",
    "baf_predictive": baf_benchmark_root / "predictive_metrics_summary.csv",
    "ieee_bootstrap": ieee_benchmark_root / "paired_bootstrap_model_differences.csv",
    "baf_bootstrap": baf_benchmark_root / "paired_bootstrap_model_differences.csv",
    "ieee_rules": find_result_file("ieee_rule_quality.csv", INPUT_ROOTS, "04_ieee_cis_ltn_rule_analysis"),
    "ieee_explanations": find_result_file("ieee_explanation_quality.csv", INPUT_ROOTS, "05_ieee_cis_rule_explanation_evaluation"),
    "ieee_ablation": find_result_file("ieee_rule_ablation.csv", INPUT_ROOTS, "06_ieee_cis_rule_ablation"),
    "baf_rules": find_result_file("baf_rule_quality.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
    "baf_explanations": find_result_file("baf_explanation_quality.csv", INPUT_ROOTS, "07_baf_ltn_generalization"),
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required upstream outputs are missing: {missing}")
input_manifest = pd.DataFrame([
    {"artifact": name, "path": str(path), "sha256": sha256_file(path)}
    for name, path in required.items()
])
display(input_manifest)
input_manifest.to_csv(output_dir / "input_artifact_manifest.csv", index=False)

## Predictive results

In [ ]:
predictive = pd.concat([
    pd.read_csv(required["ieee_predictive"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_predictive"]).assign(dataset="BAF"),
], ignore_index=True)
predictive_test = predictive.query("split == 'test'").copy()
display(predictive_test.round(4))
predictive_test.to_csv(output_dir / "table_cross_dataset_predictive_test.csv", index=False)

bootstrap = pd.concat([
    pd.read_csv(required["ieee_bootstrap"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_bootstrap"]).assign(dataset="BAF"),
], ignore_index=True)
display(bootstrap.round(5))
bootstrap.to_csv(output_dir / "table_cross_dataset_predictive_bootstrap.csv", index=False)

## Logic, explanation and ablation results

In [ ]:
rules = pd.concat([
    pd.read_csv(required["ieee_rules"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_rules"]).assign(dataset="BAF"),
], ignore_index=True)
explanations = pd.concat([
    pd.read_csv(required["ieee_explanations"]).assign(dataset="IEEE-CIS"),
    pd.read_csv(required["baf_explanations"]).assign(dataset="BAF"),
], ignore_index=True)
ablation = pd.read_csv(required["ieee_ablation"])
display(rules.round(4), explanations.round(4), ablation.round(4))
rules.to_csv(output_dir / "table_cross_dataset_rule_quality.csv", index=False)
explanations.to_csv(output_dir / "table_cross_dataset_explanation_quality.csv", index=False)
ablation.to_csv(output_dir / "table_ieee_rule_ablation.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=predictive_test, x="model", y="raw_pr_auc_mean", hue="dataset", ax=axes[0])
axes[0].set_title("Raw test PR-AUC across datasets")
axes[0].tick_params(axis="x", rotation=20)
test_rules = rules.query("split == 'test'") if "split" in rules else rules
sns.barplot(data=test_rules, x="rule", y="lift", hue="dataset", ax=axes[1])
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Test rule lift across datasets")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(output_dir / "figure_cross_dataset_summary.png", dpi=160, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
selected = predictive_test.sort_values(["dataset", "raw_pr_auc_mean"], ascending=[True, False]).groupby("dataset").head(1)
display(selected[["dataset", "model", "raw_pr_auc_mean", "fbeta_mean", "brier_mean", "ece_mean"]].round(4))
display(Markdown(
    "- Final claims must be based on the frozen artifacts and attached upstream tables listed above.\n"
    "- Predictive ranking, calibration, rule evidence and explanation quality are reported separately.\n"
    "- Cross-dataset consistency does not imply production or causal generalization."
))